In [ ]:
# Import Libraries

import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelling
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Evaluation
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix
from sklearn.calibration import calibration_curve, CalibratedClassifierCV


## Load and Understand Data Set

In [ ]:
# Loading data

df_train = pd.read_csv('/kaggle/input/playground-series-s5e12/train.csv')

df_test = pd.read_csv('/kaggle/input/playground-series-s5e12/test.csv')


In [ ]:
df_train.describe().T

In [ ]:
df_test.describe().T

## Exploratory Data Analysis

In [ ]:
# Display Histogram showing diagnosed diabetes patients
plt.figure(figsize=(8, 4))
ax = sns.countplot(data=df_train, x="diagnosed_diabetes")
plt.title("Diabetes Count")
plt.xlabel("0: Not Diabetic, 1: Diabetic")

# Calculate the total number of rows to find percentages
total = len(df_train)

# Iterate through containers and create custom labels
for container in ax.containers:
    # Generate labels: "count (percentage%)"
    labels = [f'{v.get_height()} ({v.get_height()/total:.1%})' for v in container]
    
    # Apply the custom labels to the bars
    ax.bar_label(container, labels=labels, padding=3)

plt.show()

## Define Target and Split Data

In [ ]:
TARGET = "diagnosed_diabetes"

X = df_train.drop(columns=[TARGET, "id"])
y = df_train[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

## Identify Numerical & Categorical Features

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## Preprocessing Pipeline (Shared by all models deployed)

- StandardScaler → numeric features

- OneHotEncoder → categorical features

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(
            drop="first",
            handle_unknown="ignore",
            sparse_output=True   # <<< CRITICAL
        ), categorical_features)
    ],
    sparse_threshold=1.0
)


## Defining Model 
### Model 1 - Logistic Regression

In [ ]:
logreg_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

logreg_pipe.fit(X_train, y_train)

logreg_prob = logreg_pipe.predict_proba(X_test)[:, 1]
logreg_auc = roc_auc_score(y_test, logreg_prob)

print(f"Logistic Regression ROC-AUC: {logreg_auc:.3f}")


### Model 2: Random Forest

In [ ]:
rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=50,
        max_depth=12,
        class_weight="balanced",
        random_state=42
    ))
])

rf_pipe.fit(X_train, y_train)

rf_prob = rf_pipe.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_prob)

print(f"Random Forest ROC-AUC: {rf_auc:.3f}")


### Model 3: XGBoost

In [ ]:
xgb_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="logloss",
        random_state=42
    ))
])

xgb_pipe.fit(X_train, y_train)

xgb_prob = xgb_pipe.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_prob)

print(f"XGBoost ROC-AUC: {xgb_auc:.3f}")


### Model 4: Light GBM

In [ ]:
lgbm_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.0125,
        num_leaves=71,
        subsample=0.8,
        colsample_bytree=0.8,
        force_col_wise=True,
        random_state=42
    ))
])

lgbm_pipe.fit(X_train, y_train)

lgbm_prob = lgbm_pipe.predict_proba(X_test)[:, 1]
lgbm_auc = roc_auc_score(y_test, lgbm_prob)

print(f"LightGBM ROC-AUC: {lgbm_auc:.3f}")


## Compile the Results of pipeline

In [ ]:
# Collect Results
results_df = pd.DataFrame({
    "Model": ["Logistic", "RandomForest", "XGBoost", "LightGBM"],
    "ROC_AUC": [logreg_auc, rf_auc, xgb_auc, lgbm_auc]
}).sort_values("ROC_AUC", ascending=False)

results_df

## Calibrate the Best Data

In [ ]:
# Calibrate ONLY Best Model
best_model_name = results_df.iloc[0]["Model"]

model_map = {
    "Logistic": logreg_pipe,
    "RandomForest": rf_pipe,
    "XGBoost": xgb_pipe,
    "LightGBM": lgbm_pipe
}

best_pipe = model_map[best_model_name]

### Platt Calibration

In [ ]:
calibrated = CalibratedClassifierCV(
    best_pipe,
    method="sigmoid",
    cv="prefit"
)

calibrated.fit(X_train, y_train)

cal_prob = calibrated.predict_proba(X_test)[:, 1]
cal_auc = roc_auc_score(y_test, cal_prob)

print(f"{best_model_name} + Platt ROC-AUC: {cal_auc:.3f}")

## Calibration Curve Plot

In [ ]:
frac_pos, mean_pred = calibration_curve(
    y_test, cal_prob, n_bins=10, strategy="quantile"
)

plt.figure(figsize=(6, 6))
plt.plot(mean_pred, frac_pos, marker="o")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("Predicted Probability")
plt.ylabel("Observed Fraction")
plt.title(f"Calibration Curve – {best_model_name}")
plt.grid()
plt.show()


## Prepare Test Features

In [ ]:
# Drop ID column but keep it for output
test_ids = df_test["id"]
X_test_final = df_test.drop(columns=["id"])

## Predict Diabetes Probability in Test Data

In [ ]:
test_prob = calibrated.predict_proba(X_test_final)[:, 1]

## Create Deployment Output

In [ ]:
submission = pd.DataFrame({
    "id": test_ids,
    "diabetes_probability": test_prob
})

submission.head()

## Save Output Results to File

In [ ]:
submission.to_csv("submission.csv", index=False)

print("Deployment file saved: diabetes_probability_predictions.csv")